# IRS Form 990 XML ingestion | Tax years 2023–2025

Downloads IRS publication archives (2023–2026), extracts full Form 990 diversion disclosures, and saves resumable Parquet partitions and JSON reports. **`fraud_label=1` means self-reported significant diversion of assets, not proven fraud.** Publication year is not tax year; 2025 tax-year coverage may be incomplete.

## 1. Setup

In [ ]:
from pathlib import Path
from collections import Counter
from html.parser import HTMLParser
from urllib.parse import urljoin
from xml.etree import ElementTree as ET
import json
import re
import time
import urllib.request
import zipfile

import pandas as pd
import pyarrow  # required for Parquet

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / ".gitignore").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open JupyterLab from the irs-990-fraud-detection repository")
RAW = ROOT / "data" / "raw"
STAGING = ROOT / "data" / "staging" / "form990_xml_partitions"
REPORTS = ROOT / "reports" / "ingestion"
for directory in (RAW, STAGING, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)

## 2. Configuration

In [ ]:
TARGET_TAX_YEARS = {2023, 2024, 2025}
PUBLICATION_YEARS = {2023, 2024, 2025}
INCLUDE_2026_PUBLICATIONS = True  # Turn on to improve 2025 tax-year coverage
if INCLUDE_2026_PUBLICATIONS:
    PUBLICATION_YEARS.add(2026)

RUN_MODE = "all"  # "one" or "all"
MAX_XML_PER_ARCHIVE = None  # None = every XML in selected archive; set 1000 for quick test
KEEP_DOWNLOADED_ZIPS = True
REQUEST_TIMEOUT_SECONDS = 180
RETRIES = 3

assert RUN_MODE in {"one", "all"}
print("Target tax years:", sorted(TARGET_TAX_YEARS))
print("Publication years:", sorted(PUBLICATION_YEARS))
print("Run mode:", RUN_MODE)

## 3. Find IRS archives

In [ ]:
IRS_PAGE = "https://www.irs.gov/charities-non-profits/form-990-series-downloads"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; IRS990Research/1.0)"}

class LinkCollector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            href = dict(attrs).get("href")
            if href:
                self.links.append(href)

request = urllib.request.Request(IRS_PAGE, headers=HEADERS)
with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
    page_html = response.read().decode("utf-8", errors="replace")
collector = LinkCollector()
collector.feed(page_html)

rows = []
for href in collector.links:
    url = urljoin(IRS_PAGE, href).split("?", 1)[0]
    match = re.search(r"/pub/epostcard/990/xml/(20\d{2})/[^/]+\.zip$", url, re.I)
    if match and int(match.group(1)) in PUBLICATION_YEARS:
        rows.append({"publication_year": int(match.group(1)), "url": url,
                     "archive_name": url.rsplit("/", 1)[-1]})
archives = (pd.DataFrame(rows).drop_duplicates(subset=["url"])
            .sort_values(["publication_year", "archive_name"]).reset_index(drop=True))
if archives.empty:
    raise RuntimeError("No archive URLs discovered; inspect the IRS page and URL matching rule")
print(archives.groupby("publication_year").size().rename("archive_count"))
missing_years = PUBLICATION_YEARS - set(archives.publication_year)
if missing_years:
    print("WARNING: no archives discovered for publication years:", sorted(missing_years))
display(archives.head(10))

## 4. Download and XML parsing helpers

In [ ]:
def local_name(tag):
    return tag.rsplit("}", 1)[-1].split(":")[-1]


def first_text(root, names):
    for element in root.iter():
        if local_name(element.tag) in names and element.text and element.text.strip():
            return element.text.strip()
    return None


def parse_label(value):
    if value is None:
        return None
    normalized = str(value).strip().lower()
    if normalized in {"1", "true", "yes", "y"}:
        return 1
    if normalized in {"0", "false", "no", "n"}:
        return 0
    return None


def parse_filing(root, filename, publication_year, archive_name):
    form = next((el for el in root.iter() if local_name(el.tag) == "IRS990"), None)
    if form is None:
        return None
    header = next((el for el in root.iter() if local_name(el.tag) == "ReturnHeader"), root)
    raw = first_text(form, {"MaterialDiversionOrMisuseInd"})
    ein = first_text(header, {"EIN"})
    tax_year_text = first_text(header, {"TaxYr", "TaxYear"})
    tax_year = int(tax_year_text) if tax_year_text and tax_year_text.isdigit() else None
    return {
        "publication_year": publication_year,
        "archive_name": archive_name,
        "source_xml": filename,
        "filing_id": Path(filename).stem.replace("_public", ""),
        "ein": ein.zfill(9) if ein and ein.isdigit() else ein,
        "organization_name": first_text(header, {"BusinessNameLine1Txt", "BusinessNameLine1", "BusinessName"}),
        "tax_year": tax_year,
        "tax_period_begin": first_text(header, {"TaxPeriodBeginDt", "TaxPeriodBeginDate"}),
        "tax_period_end": first_text(header, {"TaxPeriodEndDt", "TaxPeriodEndDate"}),
        "return_type": first_text(header, {"ReturnTypeCd", "ReturnType"}),
        "raw_diversion_response": raw,
        "fraud_label": parse_label(raw),
        "disclosure_field_found": raw is not None,
    }


def download_archive(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if zipfile.is_zipfile(destination):
            print("Using cached ZIP:", destination.name)
            return destination
        raise ValueError(f"Existing file is not a valid ZIP: {destination}")
    partial = destination.with_name(destination.name + ".part")
    for attempt in range(1, RETRIES + 1):
        try:
            partial.unlink(missing_ok=True)
            request = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as src, partial.open("wb") as out:
                while True:
                    chunk = src.read(1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
            if not zipfile.is_zipfile(partial):
                raise ValueError("Downloaded file is not a valid ZIP")
            partial.replace(destination)
            print(f"Downloaded {destination.name}: {destination.stat().st_size / 1024**2:.1f} MiB")
            return destination
        except Exception as exc:
            partial.unlink(missing_ok=True)
            if attempt == RETRIES:
                raise
            print(f"Download attempt {attempt} failed: {exc}; retrying")
            time.sleep(2 ** attempt)


## 5. Archive ingestion (Deflate64 supported)

In [ ]:
# STEP 5: Process XML archives (including Deflate64 ZIPs)

import subprocess
import tempfile
import os
import shutil

SEVEN_ZIP = os.environ.get("SEVEN_ZIP") or shutil.which("7zz")

OUTPUT_COLUMNS = [
    "publication_year", "archive_name", "source_xml", "filing_id", "ein",
    "organization_name", "tax_year", "tax_period_begin", "tax_period_end",
    "return_type", "raw_diversion_response", "fraud_label", "disclosure_field_found"
]


def partition_paths(archive_name):
    stem = Path(archive_name).stem
    suffix = "full" if MAX_XML_PER_ARCHIVE is None else f"first_{MAX_XML_PER_ARCHIVE}"
    return (STAGING / f"{stem}_{suffix}.parquet", REPORTS / f"{stem}_{suffix}.json")


def process_archive(archive_row):
    year = int(archive_row["publication_year"])
    name = archive_row["archive_name"]
    url = archive_row["url"]

    output, report_path = partition_paths(name)

    if output.exists() and report_path.exists():
        print("SKIP completed:", name)
        return json.loads(report_path.read_text())

    zip_path = RAW / str(year) / name
    download_archive(url, zip_path)

    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"Invalid ZIP archive: {zip_path}")

    counts = Counter()
    raw_values = Counter()
    errors = []
    records = []
    started = time.time()

    with zipfile.ZipFile(zip_path) as z:
        xml_members = [
            m for m in z.infolist()
            if m.filename.lower().endswith(".xml") and not m.is_dir()
        ]

        counts["xml_available"] = len(xml_members)
        selected_members = (
            xml_members[:MAX_XML_PER_ARCHIVE]
            if MAX_XML_PER_ARCHIVE is not None else xml_members
        )

        needs_7zip = any(
            m.compress_type not in {0, 8, 12, 14}
            for m in selected_members
        )

        # Temporary extraction is automatically cleaned up,
        # even if processing fails or is interrupted.
        with tempfile.TemporaryDirectory(prefix="irs990_") as temp_dir:

            if needs_7zip:
                if not SEVEN_ZIP:
                    raise FileNotFoundError(
                        "7-Zip not found. Install 7zz or set the SEVEN_ZIP environment variable.")
                print(f"Using 7-Zip for {name}")

                result = subprocess.run(
                    [
                        str(SEVEN_ZIP), "x", str(zip_path),
                        f"-o{temp_dir}", "-y"
                    ],
                    capture_output=True,
                    text=True
                )

                if result.returncode != 0:
                    raise RuntimeError(
                        f"7-Zip extraction failed:\n"
                        f"{result.stderr}\n{result.stdout[-2000:]}"
                    )

            for member in selected_members:
                counts["xml_attempted"] += 1

                try:
                    if needs_7zip:
                        xml_path = Path(temp_dir) / member.filename
                        root = ET.parse(xml_path).getroot()
                    else:
                        with z.open(member) as source:
                            root = ET.parse(source).getroot()

                    record = parse_filing(root, member.filename, year, name)

                    if record is None:
                        counts["excluded_non_full_990"] += 1
                        continue

                    counts["full_990_seen"] += 1

                    if record["tax_year"] not in TARGET_TAX_YEARS:
                        counts["excluded_outside_target_tax_years"] += 1
                        continue

                    counts["retained"] += 1
                    raw_values[str(record["raw_diversion_response"])] += 1
                    records.append(record)

                except (ET.ParseError, ValueError, OSError, KeyError) as exc:
                    counts["parse_errors"] += 1

                    if len(errors) < 30:
                        errors.append({
                            "source_xml": member.filename,
                            "error": str(exc)[:300]
                        })

                if counts["xml_attempted"] % 5000 == 0:
                    print(
                        f"  {name}: {counts['xml_attempted']:,} XML attempted; "
                        f"{counts['retained']:,} retained"
                    )

    frame = pd.DataFrame.from_records(records, columns=OUTPUT_COLUMNS)

    frame["fraud_label"] = pd.array(frame["fraud_label"], dtype="Int8")
    frame["tax_year"] = pd.array(frame["tax_year"], dtype="Int64")
    frame["publication_year"] = pd.array(frame["publication_year"], dtype="Int64")

    report = {
        "archive_name": name,
        "source_url": url,
        "publication_year": year,
        "target_tax_years": sorted(TARGET_TAX_YEARS),
        "is_full_archive": MAX_XML_PER_ARCHIVE is None,
        "counts": dict(counts),
        "raw_disclosure_values": dict(raw_values),
        "positive": int((frame.fraud_label == 1).sum()),
        "negative": int((frame.fraud_label == 0).sum()),
        "missing_label": int(frame.fraud_label.isna().sum()),
        "duplicate_ein_tax_year_rows": int(
            frame.duplicated(["ein", "tax_year"]).sum()
        ),
        "errors_first_30": errors,
        "seconds_elapsed": round(time.time() - started, 1),
        "parquet_path": str(output.relative_to(ROOT)),
    }

    tmp_parquet = output.with_name(output.name + ".part")
    tmp_report = report_path.with_name(report_path.name + ".part")

    try:
        frame.to_parquet(tmp_parquet, index=False)
        tmp_report.write_text(json.dumps(report, indent=2))
        tmp_parquet.replace(output)
        tmp_report.replace(report_path)
    finally:
        tmp_parquet.unlink(missing_ok=True)
        tmp_report.unlink(missing_ok=True)

    if not KEEP_DOWNLOADED_ZIPS:
        zip_path.unlink()

    print(
        f"SAVED {name}: {len(frame):,} retained; "
        f"{report['positive']} yes; {report['negative']} no; "
        f"{report['missing_label']} missing"
    )

    return report

## 6. Run or resume ingestion

Completed archives are skipped. The Deflate64 fallback temporarily extracts files with 7-Zip and removes them afterward. Keep enough disk space for extracted files.

In [ ]:
work = archives.copy()
if RUN_MODE == "one":
    cached = work[work.archive_name.eq("2025_TEOS_XML_01A.zip")]
    work = cached.head(1) if not cached.empty else work.head(1)
print(f"Archives selected: {len(work)}")

run_reports = []
for _, archive_row in work.iterrows():
    try:
        run_reports.append(process_archive(archive_row))
    except Exception as exc:
        print(f"FAILED {archive_row['archive_name']}: {type(exc).__name__}: {exc}")
        # Stop rather than silently marking an archive complete.
        raise
print("Processed or skipped:", len(run_reports))

## 7. Review archive reports

In [ ]:
report_files = sorted(REPORTS.glob("*_full.json" if MAX_XML_PER_ARCHIVE is None else f"*_first_{MAX_XML_PER_ARCHIVE}.json"))
reports = [json.loads(path.read_text()) for path in report_files]
report_df = pd.DataFrame([{
    "archive_name": r["archive_name"],
    "publication_year": r["publication_year"],
    "retained": r["counts"].get("retained", 0),
    "positive": r["positive"],
    "negative": r["negative"],
    "missing": r["missing_label"],
    "parse_errors": r["counts"].get("parse_errors", 0),
} for r in reports])
if report_df.empty:
    print("No partitions yet. Run Step 6.")
else:
    display(report_df)
    print("Totals:")
    display(report_df[["retained", "positive", "negative", "missing", "parse_errors"]].sum().to_frame("total"))
    print(f"Completed archives: {len(report_df)} / {len(archives)} discovered")
    if len(report_df) < len(archives):
        print("NOTE: Partial archive coverage; these counts are not full-population prevalence estimates.")

## 8. Review tax-year coverage and duplicates

In [ ]:
partition_files = sorted(STAGING.glob("*_full.parquet" if MAX_XML_PER_ARCHIVE is None else f"*_first_{MAX_XML_PER_ARCHIVE}.parquet"))
if not partition_files:
    print("No Parquet partitions yet.")
else:
    filings = pd.concat((pd.read_parquet(p) for p in partition_files), ignore_index=True)
    print("Total retained filing records:", f"{len(filings):,}")
    print("Unique source filings:", f"{filings['filing_id'].nunique():,}")
    print("Missing EIN:", filings.ein.isna().sum())
    print("Missing labels:", filings.fraud_label.isna().sum())
    by_year = (filings.groupby("tax_year", dropna=False)
               .agg(filings=("filing_id", "size"),
                    positive=("fraud_label", lambda s: int((s == 1).sum())),
                    negative=("fraud_label", lambda s: int((s == 0).sum())),
                    missing=("fraud_label", lambda s: int(s.isna().sum())))
               .reset_index())
    by_year["positive_rate_labeled"] = by_year["positive"] / (by_year["positive"] + by_year["negative"]).replace(0, float("nan"))
    display(by_year)
    repeated = filings[filings.duplicated(["ein", "tax_year"], keep=False)].copy()
    print("Rows in repeated EIN + tax year groups:", len(repeated))
    if not repeated.empty:
        display(repeated[["ein", "tax_year", "filing_id", "archive_name", "fraud_label"]].sort_values(["ein", "tax_year"]).head(30))
        disagreements = (repeated.groupby(["ein", "tax_year"], dropna=False)["fraud_label"]
                         .nunique(dropna=True).gt(1).sum())
        print("Groups with conflicting nonmissing disclosure labels:", disagreements)
    print("WARNING: 2025 tax-year coverage is incomplete unless later publication archives are included and assessed.")

## 9. Save ingestion manifest

In [ ]:
manifest = {
    "source_page": IRS_PAGE,
    "target_tax_years": sorted(TARGET_TAX_YEARS),
    "publication_years_requested": sorted(PUBLICATION_YEARS),
    "run_mode": RUN_MODE,
    "max_xml_per_archive": MAX_XML_PER_ARCHIVE,
    "discovered_archives": len(archives),
    "completed_partitions_in_current_mode": len(report_files),
    "completed_archive_names": [r["archive_name"] for r in reports],
    "fraud_label_xml_tag": "MaterialDiversionOrMisuseInd",
    "fraud_label_definition": "Self-reported significant diversion of assets; not confirmed organizational fraud",
    "limitations": [
        "Publication year differs from tax year",
        "2025 tax-year filings may require 2026 or later publication archives",
        "Duplicate EIN + tax year filings retained pending resolution",
        "A No disclosure is not proof that fraud did not occur",
    ],
}
manifest_path = REPORTS / "01_ingestion_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print("Saved:", manifest_path)